In [3]:
# This code will work on Jyupyter Notes only.
# To make run on Colab, you need to connect to remotely hosted llms

import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_community.vectorstores import FAISS

# MODERN CORE IMPORTS: Zero reliance on legacy chain wrappers
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    """Helper function to cleanly join retrieved document pieces."""
    return "\n\n".join(doc.page_content for doc in docs)

def setup_local_rag(data_folder_path: str):
    # 1. Load banking and financial PDFs from the local folder
    print("Loading financial documents...")
    if not os.path.exists(data_folder_path):
        os.makedirs(data_folder_path)
        print(f"Created folder '{data_folder_path}'. Please add financial PDFs to it.")
        return None

    loader = DirectoryLoader(data_folder_path, glob="**/*.txt", loader_cls=TextLoader)
    documents = loader.load()

    if not documents:
        print("No PDF documents found in the specified folder.")
        return None

    # 2. Split financial text into high-precision chunks
    print("Splitting documents into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100
    )
    chunks = text_splitter.split_documents(documents)

    # 3. Initialize local embeddings via Ollama
    print("Generating embeddings and building FAISS index...")
    embeddings = OllamaEmbeddings(model="llama3.2")

    # 4. Ingest financial records into FAISS Vector Database
    vector_store = FAISS.from_documents(chunks, embeddings)
    retriever = vector_store.as_retriever(search_kwargs={"k": 4})

    # 5. Initialize local Llama 3.2 LLM (Temperature set to 0 for factual accuracy)
    llm = OllamaLLM(model="llama3.2", temperature=0)

    # 6. Define a finance-domain strict prompt template
    system_prompt = (
        "You are an expert financial analyst and compliance auditor assistant. "
        "Use the provided pieces of retrieved context to accurately answer the financial query. "
        "Adhere to the following rules strictly:\n"
        "1. Base your answer ONLY on the retrieved context below.\n"
        "2. If the context contains monetary figures, percentages, or ratios, quote them exactly.\n"
        "3. If the answer cannot be verified directly by the context, state 'Information not found in available documents.' Do not extrapolate.\n\n"
        "Context:\n{context}\n\n"
        "Question: {input}\n"
        "Answer:"
    )
    prompt = ChatPromptTemplate.from_template(system_prompt)

    # 7. Create the RAG chain using explicit LCEL pipes
    # This completely bypasses 'create_retrieval_chain' and 'create_stuff_documents_chain'
    rag_chain = (
        #{"context": retriever | format_docs, "input": RunnablePassthrough()}
        {"context": retriever , "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

# --- Execution ---
if __name__ == "__main__":
    FOLDER_PATH = "./financial_knowledge_base"
    rag_pipeline = setup_local_rag(FOLDER_PATH)

    if rag_pipeline:
        print("\n--- Running Banking & Finance Analysis ---")

        # USE CASE 1: Knowing about the document
        query_1 = "What is the document is about?"
        print(f"\n[Use Case 1 - ] Query: {query_1}")
        response_1 = rag_pipeline.invoke(query_1) # LCEL accepts string directly here
        print(f"Response: {response_1}\n")

        # USE CASE 2: Besel function use case
        query_2 = "How besel function helps in banking?"
        print(f"\n[Use Case 2 - Compliance Audit] Query: {query_2}")
        response_2 = rag_pipeline.invoke(query_2)
        print(f"Response: {response_2}\n")


ModuleNotFoundError: No module named 'langchain_community'